# 3. 데이터 분할 및 기준선

서울시 따릉이 일별 수요 예측. [요구사항](../../docs/ml/REQUIREMENTS.md).

In [1]:
from pathlib import Path
import sys, json, time, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))
from src.ml.features import FEATURES, load_daily, make_features
from src.ml.splits import split_data, time_cv
from src.ml.evaluation import metrics, compare, grouped_metrics
from sklearn.base import clone
from sklearn.inspection import permutation_importance

plt.rcParams.update({"figure.figsize": (10, 4), "font.family": "Malgun Gothic", "axes.unicode_minus": False})
OUT = ROOT / "outputs/ml"
OUT.mkdir(parents=True, exist_ok=True)
LOCAL = ROOT / "data/processed"
LOCAL.mkdir(parents=True, exist_ok=True)
daily = load_daily(ROOT)
frame = make_features(daily)
train, validation, final_train, test = split_data(frame)


In [2]:
split_summary = pd.DataFrame([
    {"split": name, "start": f"{part.date.min():%Y-%m-%d}", "end": f"{part.date.max():%Y-%m-%d}", "rows": len(part)}
    for name, part in [("train", train), ("validation", validation), ("final_train", final_train), ("test", test)]
])
split_summary.to_csv(OUT / "03_splits.csv", index=False)
display(split_summary)
baselines = {"train_mean": np.full(len(validation), train.rentals.mean()), "lag_7": validation.lag_7.to_numpy()}
baseline_metrics = compare(validation, baselines)
baseline_metrics.to_csv(OUT / "03_baselines.csv")
display(baseline_metrics.round(4))


,split,start,end,rows
0,train,2025-01-30,2025-09-29,243
1,validation,2025-10-01,2025-12-31,92
2,final_train,2025-01-30,2025-12-30,335
3,test,2026-01-01,2026-06-30,181


,MAE,RMSE,WAPE,R2
model,,,,
train_mean,33559.6833,41662.5338,0.3607,-0.2307
lag_7,27382.0870,41039.1056,0.2943,-0.1942


- 검증 MAE는 학습 평균 33,560건, 전주 동일 요일 27,382건임.
- 두 기준선 중 lag_7의 검증 MAE가 더 낮음.
- 학습 끝과 검증 시작 사이 하루를 제외해 타깃 가용 시점을 맞춤.
- 테스트 점수는 아직 계산하지 않음.